# Пример 6 из 6: публичный бенчмарк (τ²-bench) — тема 15

Кроме собственной корзины задач полезно прогонять агента на **публичном эталонном бенчмарке** — это готовый источник задач для оценки **способностей** (capability eval). Здесь подключаем **τ²-bench** (открытый бенчмарк диалоговых tool-агентов, домены airline/retail/telecom) и запускаем на нём агента через тот же OpenAI-совместимый эндпоинт.

τ²-бенчмарк **многоходовой** (агент + симулятор пользователя + проверка итогового состояния БД), поэтому он показывает и pass@k / pass^k, и стоимость. Он **намеренно трудный** — единичный успех не означает надёжности.

> Внешний open-source бенчмарк (лицензия MIT). Ставится из GitHub; на первом запуске скачивается ~33 МБ данных домена. Это **дополнительный/повышенный** пример: тяжелее и дороже базовых.

## Предустановка

Пакет ставится из GitHub (в PyPI одноимённого нет):

```bash
pip install "git+https://github.com/sierra-research/tau2-bench.git"
```

Он тянет зависимость **litellm**, через которую можно ходить в любой OpenAI-совместимый эндпоинт.

In [1]:
import os, subprocess, contextlib, io
from pathlib import Path
import pandas as pd

# tau2-bench использует litellm; направляем его в наш OpenAI-совместимый эндпоинт.
os.environ.setdefault("OPENAI_API_KEY", os.environ.get("LLM_API_KEY", ""))
os.environ.setdefault("OPENAI_API_BASE", os.environ.get("LLM_BASE_URL", ""))
os.environ.setdefault("OPENAI_BASE_URL", os.environ.get("LLM_BASE_URL", ""))
AGENT_MODEL = os.environ.get("LLM_MODEL", "openai/gpt-4.1-mini")   # litellm-имя (провайдер openai)

try:
    import tau2
    print("tau2-bench установлен.")
except ImportError:
    print('Установите: pip install "git+https://github.com/sierra-research/tau2-bench.git"')

2026-07-20 01:42:06.849 | WARNING  | tau2.utils.utils:<module>:15 - No .env file found


2026-07-20 01:42:06.849 | INFO     | tau2.utils.utils:<module>:23 - Using data directory from environment: <TAU2_DATA_DIR>


2026-07-20 01:42:08.477 | INFO     | tau2.utils.llm_utils:<module>:102 - LiteLLM: Cache is disabled


2026-07-20 01:42:08.551 | DEBUG    | tau2.registry:<module>:281 - Registering default components...


2026-07-20 01:42:08.553 | DEBUG    | tau2.registry:<module>:292 - Voice dependencies not installed, skipping voice user registration


2026-07-20 01:42:08.553 | DEBUG    | tau2.registry:<module>:350 - Default components registered successfully. Registry info: {
  "domains": [
    "mock",
    "airline",
    "retail",
    "telecom",
    "telecom-workflow",
    "banking_knowledge"
  ],
  "agents": [
    "llm_agent",
    "llm_agent_gt",
    "llm_agent_solo",
    "discrete_time_audio_native_agent"
  ],
  "users": [
    "user_simulator",
    "dummy_user"
  ],
  "task_sets": [
    "mock",
    "airline",
    "retail",
    "telecom_full",
    "telecom_small",
    "telecom",
    "telecom-workflow",
    "banking_knowledge"
  ]
}


tau2-bench установлен.


## Данные бенчмарка

Данные доменов лежат в репозитории. Подтягиваем **только домен retail** и промпты симулятора пользователя (sparse-checkout, ~33 МБ), путь задаём через `TAU2_DATA_DIR`.

In [2]:
DOMAIN = "retail"
def ensure_data():
    dd = os.environ.get("TAU2_DATA_DIR")
    if dd and (Path(dd) / f"tau2/domains/{DOMAIN}/tasks.json").exists():
        return dd
    src = Path(".tau2_src")
    if not (src / f"data/tau2/domains/{DOMAIN}/tasks.json").exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                        "https://github.com/sierra-research/tau2-bench.git", str(src)], check=True)
        subprocess.run(["git", "-C", str(src), "sparse-checkout", "set",
                        f"data/tau2/domains/{DOMAIN}", "data/tau2/user_simulator"], check=True)
    os.environ["TAU2_DATA_DIR"] = str((src / "data").resolve())
    return os.environ["TAU2_DATA_DIR"]

ensure_data()
from tau2.run import get_tasks
tasks = get_tasks(DOMAIN, num_tasks=2)
print(f"Домен {DOMAIN}: доступно задач (первые 2): {[t.id for t in tasks]}")

Домен retail: доступно задач (первые 2): ['0', '1']


## Прогон агента на бенчмарке (несколько попыток на задачу)

Запускаем агента `openai/gpt-4.1-mini` на 2 задачах по `k=3` попытки. `run_domain` сам поднимает изолированную среду, симулятор пользователя и грейдер итогового состояния БД (reward 1.0 = задача решена).

In [3]:
from tau2.run import TextRunConfig, run_domain

K = 3
cfg = TextRunConfig(domain=DOMAIN, num_tasks=2, num_trials=K,
                    llm_agent=AGENT_MODEL, llm_user=AGENT_MODEL,
                    llm_args_agent={"temperature": 0.0}, llm_args_user={"temperature": 0.0},
                    max_concurrency=3, log_level="ERROR")
with contextlib.redirect_stdout(io.StringIO()):        # прячем большую rich-панель
    res = run_domain(cfg)

rows = [{"task": s.task_id, "reward": round(s.reward_info.reward, 2)} for s in res.simulations]
df = pd.DataFrame(rows).sort_values("task")
print(df.to_string(index=False))

╭─────────────────────────────────────────── Simulation Configuration ────────────────────────────────────────────╮
│ Domain: retail  Task Set: Default  Tasks: All                                                                   │
│ Trials: 3  Max Steps: 200  Max Errors: 10                                                                       │
│                                                                                                                 │
│ Agent: llm_agent → openai/gpt-4.1-mini                                                                          │
│ User:  user_simulator → openai/gpt-4.1-mini                                                                     │
│                                                                                                                 │
│ Save: Not specified  Concurrency: 3  Verbose: False                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

0/2 (trial 1/3). Running task 0, trial 1

1/2 (trial 1/3). Running task 1, trial 1

0/2 (trial 2/3). Running task 0, trial 2

Status: 0/6 complete. Avg reward: N/A. 3 running: 0.0(30s), 1.0(30s), 0.1(30s)

...

╭────────────────────────────────────────────── Simulation Overview ──────────────────────────────────────────────╮
│ Task ID: 0                                                                                                      │
│ Trial: 2                                                                                                        │
│ Duration: 49.80s                                                                                                │
│ Mode: half_duplex                                                                                               │
│ Termination Reason: TerminationReason.USER_STOP                                                                 │
│ Agent Cost: $0.0085                                                                                             │
│ User Cost: $0.0024                                                                                              │
│ Reward: ✅ 1.0000 (DB: 1.0, NL_ASSERTION: 1.0)                                                                  │
│                                                                                                                 │
│ DB Check:✅ 1.0                                                                                                 │
│                                                                                                                 │
│ Action Checks:                                                                                                  │
│ - 0: agent find_user_id_by_name_zip [read] ✅ 1.0                                                               │
│ - 1: agent get_order_details [read] ✅ 1.0                                                                      │
│ - 2: agent get_product_details [read] ✅ 1.0                                                                    │
│ - 3: agent get_product_details [read] ✅ 1.0                                                                    │
│ - 4: agent exchange_delivered_order_items [write] ✅ 1.0                                                        │
│                                                                                                                 │
│ Partial Action Reward: 5/5 (100.0%)                                                                             │
│   Read:  4/4 (100.0%)                                                                                           │
│   Write: 1/1 (100.0%)                                                                                           │
│                                                                                                                 │
│ Additional Info:                                                                                                │
│ env: None                                                                                                       │
│ nl: {'note': 'No nl_assertions to evaluate'}                                                                    │
│ communicate: {'note': 'No communicate_info to evaluate'}                                                        │
│ action: None                                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Simulation Overview ──────────────────────────────────────────────╮
│ Task ID: 1                                                                                                      │
│ Trial: 2                                                                                                        │
│ Duration: 52.09s                                                                                                │
│ Mode: half_duplex                                                                                               │
│ Termination Reason: TerminationReason.USER_STOP                                                                 │
│ Agent Cost: $0.0116                                                                                             │
│ User Cost: $0.0026                                                                                              │
│ Reward: ✅ 1.0000 (DB: 1.0, NL_ASSERTION: 1.0)                                                                  │
│                                                                                                                 │
│ DB Check:✅ 1.0                                                                                                 │
│                                                                                                                 │
│ Action Checks:                                                                                                  │
│ - 0: agent find_user_id_by_name_zip [read] ✅ 1.0                                                               │
│ - 1: agent get_order_details [read] ✅ 1.0                                                                      │
│ - 2: agent get_product_details [read] ✅ 1.0                                                                    │
│ - 3: agent get_product_details [read] ✅ 1.0                                                                    │
│ - 4: agent exchange_delivered_order_items [write] ✅ 1.0                                                        │
│                                                                                                                 │
│ Partial Action Reward: 5/5 (100.0%)                                                                             │
│   Read:  4/4 (100.0%)                                                                                           │
│   Write: 1/1 (100.0%)                                                                                           │
│                                                                                                                 │
│ Additional Info:                                                                                                │
│ env: None                                                                                                       │
│ nl: {'note': 'No nl_assertions to evaluate'}                                                                    │
│ communicate: {'note': 'No communicate_info to evaluate'}                                                        │
│ action: None                                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Successfully completed all simulations!
To review the simulations, run: tau2 view

╭─────────────────────────────────────────── Agent Performance Metrics ───────────────────────────────────────────╮
│   ═══ Overview ═══                                                                                              │
│   Total Simulations         6                                                                                   │
│   Total Tasks               2                                                                                   │
│                                                                                                                 │
│   ═══ Reward Metrics ═══                                                                                        │
│   🏆 Average Reward         0.5000                                                                              │
│      Pass^1                 0.500                                                                               │
│      Pass^2                 0.167                                                                               │
│      Pass^3                 0.000                                                                               │
│   💰 Avg Cost/Conversation  $0.0082                                                                             │
│                                                                                                                 │
│   ═══ Action Metrics ═══                                                                                        │
│   📖 Read Actions           24/24 (100.0%)                                                                      │
│   ✏️  Write Actions         3/6 (50.0%)                                                                         │
│                                                                                                                 │
│   ═══ DB Match ═══                                                                                              │
│   🗄️  DB Match              ✓ 3 / ✗ 3 (50.0%)                                                                   │
│                                                                                                                 │
│   ═══ Authentication ═══                                                                                        │
│      Not Checked            6                                                                                   │
│                                                                                                                 │
│   ═══ Termination ═══                                                                                           │
│   🛑 Normal Stop            6 (👤 6 / 🤖 0)                                                                     │
│                                                                                                                 │
│   ═══ LLM Judge Review ═══                                                                                      │
│   🤖 Agent Errors           0 errors                                                                            │
│      Sims by severity       -                                                                                   │
│   👤 User Errors            0 errors                                                                            │
│      Sims by severity       -                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

task  reward
   0     0.0
   0     1.0
   0     1.0
   1     0.0
   1     0.0
   1     1.0


## Метрики стабильности pass@k / pass^k

Для каждой задачи по `k` попыткам считаем **pass@k** (успех хотя бы раз) и **pass^k** (успех во всех попытках). На трудном публичном бенчмарке pass^k обычно низкий — это и есть сигнал ненадёжности.

In [4]:
def success(r): return r >= 0.999
agg = []
for task, g in df.groupby("task"):
    ok = [success(r) for r in g["reward"]]
    agg.append({"task": task, "k": len(ok), "успехов": sum(ok),
                "pass@k": int(any(ok)), "pass^k": int(all(ok))})
res_df = pd.DataFrame(agg)
print(res_df.to_string(index=False))
print(f"\nСредний pass@k = {res_df['pass@k'].mean():.2f}, средний pass^k = {res_df['pass^k'].mean():.2f}")

task  k  успехов  pass@k  pass^k
   0  3        2       1       0
   1  3        1       1       0

Средний pass@k = 1.00, средний pass^k = 0.00


## Итоги

- Публичный бенчмарк (**τ²-bench**) — готовый источник задач для **capability eval**; подключается к тому же OpenAI-совместимому эндпоинту через litellm.
- Бенчмарк многоходовой и проверяет **итоговое состояние среды** (как и наша собственная корзина) — это подтверждает, что оценивать нужно траекторию и результат, а не текст.
- На трудном бенчмарке **pass^k** закономерно низкий: единичный успех ≠ надёжность. Публичные наборы дополняют собственную корзину, но не заменяют её (она ближе к вашему продакшену).

На этом набор примеров модуля завершён: собственная корзина + harness + грейдеры + LLM-судья + регрессия + публичный бенчмарк.